# Notebook 05-UNSW — Explanation Stability Under Perturbation

**Project:** Calibrated and Stability-Aware Explainable Intrusion Detection  
**Author:** Md Anas Biswas, University of Portsmouth  
**Contribution:** C3 — Explanation stability under Gaussian, FGSM, and PGD perturbation

## What this notebook does

1. Builds three perturbed versions of the SHAP eval subsample (the same 2000 samples used in Notebook 04):
   - **Gaussian noise** (σ = 0.05 in standardised feature space)
   - **FGSM** (single-step adversarial attack, ε = 0.05, crafted against the DNN)
   - **PGD** (10-step adversarial attack, ε = 0.05, α = 0.01, crafted against the DNN)
2. Re-computes SHAP for all 6 models on each perturbed input set
3. Computes three stability metrics per (model, perturbation):
   - **Jaccard top-10** — fraction of top-10 features that survive perturbation
   - **Lipschitz median** — median ||SHAP_perturbed − SHAP_original||₂ / ||x_perturbed − x_original||₂
   - **F-Fidelity** — does masking the top-k features actually reduce prediction confidence?
4. Saves results, generates heatmaps and comparison tables
5. Commits and pushes

## Perturbation crafting strategy

FGSM and PGD are gradient-based attacks that require model gradients. The DNN has gradients; the tree models do not. We craft the adversarial perturbations **against the DNN** (specifically `dnn_5class_smote`), then transfer them to RF and XGBoost. This is standard practice in the adversarial robustness literature — it tests whether tree-model explanations are robust to perturbations specifically designed to mislead a neural network.

## SHAP recomputation strategy

We need to re-run SHAP on each perturbed input. To keep this tractable:
- **RF/XGB:** re-run on all 2000 samples (fast — under 5 min each)
- **DNN:** subsample to 500 perturbed samples for KernelExplainer (each full run takes 15–20 min, doing it 3× would be 45–60 min per DNN). 500 stratified samples gives meaningful coverage with manageable runtime.

## Runtime budget

- Perturbation generation: ~2 min
- RF SHAP × 2 × 3 perturbations: ~25 min
- XGB SHAP × 2 × 3 perturbations: ~3 min
- DNN SHAP × 2 × 3 perturbations (500 samples each): ~30 min
- Metric computation + figures + commit: ~5 min
- **Total: ~60–75 min**

## 1. Environment setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os, shutil
shutil.copy('/content/drive/MyDrive/XIDS_Research/.gitconfig',       '/root/.gitconfig')
shutil.copy('/content/drive/MyDrive/XIDS_Research/.git-credentials', '/root/.git-credentials')

PROJECT_DIR = '/content/drive/MyDrive/XIDS_Research/xids-research'
os.chdir(PROJECT_DIR)

!git config --global credential.helper 'store --file /content/drive/MyDrive/XIDS_Research/.git-credentials'
!git pull origin main

!pip install -q shap

import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'\nDevice: {device}')

From https://github.com/anasbiswas1/xids-research
 * branch            main       -> FETCH_HEAD
Already up to date.

Device: cuda


## 2. Load data, models, SHAP subsample indices

In [3]:
import numpy as np
import json
import joblib
from pathlib import Path

REPO       = Path(PROJECT_DIR)
PROC_DIR   = REPO / 'data/processed/unsw_nb15'
MODELS_DIR = REPO / 'models/unsw_nb15'
SHAP_DIR   = REPO / 'shap_values/unsw_nb15'
STAB_DIR   = SHAP_DIR / 'stability'
STAB_DIR.mkdir(parents=True, exist_ok=True)

X_train = np.load(PROC_DIR / 'X_train.npy')
X_test  = np.load(PROC_DIR / 'X_test.npy')
y_test_binary = np.load(PROC_DIR / 'y_test_binary.npy')
y_test_5class = np.load(PROC_DIR / 'y_test_5class.npy')

with open(PROC_DIR / 'feature_names.json') as f:
    feature_names = json.load(f)

shap_idx = np.load(SHAP_DIR / 'shap_subsample_idx.npy')
X_shap = X_test[shap_idx]
y_shap_binary = y_test_binary[shap_idx]
y_shap_5class = y_test_5class[shap_idx]

FIVE_CLASS_NAMES = ['Normal', 'DoS', 'Probe', 'R2L', 'U2R']
N_FEATURES = X_train.shape[1]

print(f'SHAP subsample: {X_shap.shape}')
print(f'Total features: {N_FEATURES}')

SHAP subsample: (2000, 196)
Total features: 196


## 3. Generate perturbed inputs

All three perturbations operate in **standardised feature space** (the same space the models see). ε = 0.05 means the perturbation magnitude is 5% of one standard deviation per feature — small enough to be subtle, large enough to challenge the model.

In [4]:
EPSILON = 0.05  # consistent with NSL-KDD and CIC-IDS2017
PGD_ALPHA = 0.01
PGD_STEPS = 10

# --- Gaussian noise ---
rng = np.random.default_rng(42)
noise = rng.normal(0, EPSILON, size=X_shap.shape).astype(np.float32)
X_gaussian = (X_shap + noise).astype(np.float32)

print(f'Gaussian noise (σ={EPSILON}):')
print(f'  Mean ||noise||_2 per sample: {np.linalg.norm(noise, axis=1).mean():.4f}')
print(f'  Sample-wise max deviation:   {np.abs(noise).max():.4f}')

Gaussian noise (σ=0.05):
  Mean ||noise||_2 per sample: 0.6995
  Sample-wise max deviation:   0.2504


In [5]:
# --- FGSM and PGD against the 5-class DNN ---
import torch.nn as nn
import torch.nn.functional as F

class XIDSMLP(nn.Module):
    def __init__(self, n_features, n_classes, p_drop=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(p_drop),
            nn.Linear(256, 128),        nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(p_drop),
            nn.Linear(128, 64),         nn.BatchNorm1d(64),  nn.ReLU(), nn.Dropout(p_drop),
            nn.Linear(64, 32),          nn.BatchNorm1d(32),  nn.ReLU(), nn.Dropout(p_drop),
            nn.Linear(32, n_classes),
        )
    def forward(self, x):
        return self.net(x)

dnn5 = XIDSMLP(N_FEATURES, 5).to(device)
dnn5.load_state_dict(torch.load(MODELS_DIR / 'dnn_5class_smote.pt', map_location=device))
dnn5.eval()

def fgsm_attack(x_np, y_np, model, eps):
    """Single-step FGSM: x_adv = x + ε · sign(∇_x L(model(x), y))."""
    x = torch.tensor(x_np, dtype=torch.float32, device=device, requires_grad=True)
    y = torch.tensor(y_np, dtype=torch.long, device=device)
    logits = model(x)
    loss = F.cross_entropy(logits, y)
    loss.backward()
    x_adv = x + eps * x.grad.sign()
    return x_adv.detach().cpu().numpy().astype(np.float32)

def pgd_attack(x_np, y_np, model, eps, alpha, n_steps):
    """PGD: iterative FGSM with projection back into ε-ball."""
    x_orig = torch.tensor(x_np, dtype=torch.float32, device=device)
    y      = torch.tensor(y_np, dtype=torch.long,    device=device)
    x_adv  = x_orig.clone().detach()
    # Optional small random init within ε-ball
    x_adv += torch.empty_like(x_adv).uniform_(-eps, eps)
    for _ in range(n_steps):
        x_adv.requires_grad_(True)
        logits = model(x_adv)
        loss = F.cross_entropy(logits, y)
        grad = torch.autograd.grad(loss, x_adv)[0]
        x_adv = x_adv.detach() + alpha * grad.sign()
        # Project back into the ε-ball around x_orig
        delta = torch.clamp(x_adv - x_orig, -eps, eps)
        x_adv = (x_orig + delta).detach()
    return x_adv.cpu().numpy().astype(np.float32)

# Generate in batches to manage GPU memory
BATCH = 256
X_fgsm = np.zeros_like(X_shap, dtype=np.float32)
X_pgd  = np.zeros_like(X_shap, dtype=np.float32)

for i in range(0, len(X_shap), BATCH):
    xb = X_shap[i:i+BATCH]
    yb = y_shap_5class[i:i+BATCH]
    X_fgsm[i:i+BATCH] = fgsm_attack(xb, yb, dnn5, EPSILON)
    X_pgd[i:i+BATCH]  = pgd_attack(xb, yb, dnn5, EPSILON, PGD_ALPHA, PGD_STEPS)

print(f'FGSM:')
print(f'  Mean ||δ||_2 per sample: {np.linalg.norm(X_fgsm - X_shap, axis=1).mean():.4f}')
print(f'  Sample-wise max deviation: {np.abs(X_fgsm - X_shap).max():.4f}  (target ≤ {EPSILON})')
print(f'PGD:')
print(f'  Mean ||δ||_2 per sample: {np.linalg.norm(X_pgd - X_shap, axis=1).mean():.4f}')
print(f'  Sample-wise max deviation: {np.abs(X_pgd - X_shap).max():.4f}  (target ≤ {EPSILON})')

FGSM:
  Mean ||δ||_2 per sample: 0.7000
  Sample-wise max deviation: 0.0500  (target ≤ 0.05)
PGD:
  Mean ||δ||_2 per sample: 0.6123
  Sample-wise max deviation: 0.0500  (target ≤ 0.05)


In [6]:
# Save perturbed inputs
np.save(STAB_DIR / 'X_gaussian.npy', X_gaussian)
np.save(STAB_DIR / 'X_fgsm.npy',     X_fgsm)
np.save(STAB_DIR / 'X_pgd.npy',      X_pgd)

PERTURBATIONS = {
    'gaussian': X_gaussian,
    'fgsm':     X_fgsm,
    'pgd':      X_pgd,
}
print('Perturbations saved.')

Perturbations saved.


## 4. Confirm attack effectiveness

Quick sanity check: did the adversarial attacks actually fool the DNN? If accuracy doesn't drop under PGD, the attack isn't strong enough.

In [7]:
def dnn_acc(model, X, y):
    with torch.no_grad():
        x = torch.tensor(X, dtype=torch.float32, device=device)
        pred = model(x).argmax(1).cpu().numpy()
    return (pred == y).mean()

print('Attack effectiveness (5-class DNN accuracy):')
print(f'  Clean:    {dnn_acc(dnn5, X_shap, y_shap_5class):.4f}')
print(f'  Gaussian: {dnn_acc(dnn5, X_gaussian, y_shap_5class):.4f}')
print(f'  FGSM:     {dnn_acc(dnn5, X_fgsm, y_shap_5class):.4f}')
print(f'  PGD:      {dnn_acc(dnn5, X_pgd, y_shap_5class):.4f}')
print('\nExpected pattern: Clean > Gaussian > FGSM > PGD (PGD most effective).')

Attack effectiveness (5-class DNN accuracy):
  Clean:    0.6410
  Gaussian: 0.5980
  FGSM:     0.4690
  PGD:      0.4275

Expected pattern: Clean > Gaussian > FGSM > PGD (PGD most effective).


## 5. SHAP recomputation helpers (same as Notebook 04)

In [8]:
import shap
import time
import xgboost as xgb

def shap_rf(model, X):
    explainer = shap.TreeExplainer(model)
    sv = explainer.shap_values(X)
    if isinstance(sv, list):
        sv = np.stack(sv, axis=-1)
    elif sv.ndim == 3 and sv.shape[0] != X.shape[0]:
        sv = np.transpose(sv, (1, 2, 0))
    elif sv.ndim == 2:
        sv = sv[..., np.newaxis]
    return sv

def shap_xgb(model, X, n_classes):
    dmat = xgb.DMatrix(X)
    booster = model.get_booster()
    contribs = booster.predict(dmat, pred_contribs=True)
    if n_classes == 2:
        sv = contribs[:, :-1]
        return np.stack([-sv, sv], axis=-1)
    contribs = contribs[:, :, :-1]
    return np.transpose(contribs, (0, 2, 1))

def shap_dnn(state_dict, X_shap_in, X_bg, n_features, n_classes):
    model = XIDSMLP(n_features, n_classes).to(device)
    model.load_state_dict(state_dict)
    model.eval()
    def predict_proba(arr):
        with torch.no_grad():
            t = torch.tensor(arr, dtype=torch.float32, device=device)
            return F.softmax(model(t), dim=1).cpu().numpy()
    explainer = shap.KernelExplainer(predict_proba, X_bg)
    sv = explainer.shap_values(X_shap_in, nsamples='auto', silent=True)
    if isinstance(sv, list):
        sv = np.stack(sv, axis=-1)
    return sv

print('Helpers ready.')

Helpers ready.


In [9]:
import psutil, os
print(f'RAM used: {psutil.virtual_memory().used / 1024**3:.1f} GB')
print(f'RAM free: {psutil.virtual_memory().available / 1024**3:.1f} GB')

# How big are the RF models?
from pathlib import Path
m = Path('/content/drive/MyDrive/XIDS_Research/xids-research/models/unsw_nb15')
for f in sorted(m.glob('rf_*.joblib')):
    print(f'{f.name}: {f.stat().st_size / 1024**2:.1f} MB')

# How many trees and what depth?
import joblib
rf = joblib.load(m / 'rf_binary_cw.joblib')
depths = [t.get_depth() for t in rf.estimators_]
print(f'\nRF binary: {len(rf.estimators_)} trees, depth range {min(depths)}-{max(depths)}, mean {sum(depths)/len(depths):.1f}')

RAM used: 1.8 GB
RAM free: 10.6 GB
rf_5class_smote.joblib: 682.1 MB
rf_binary_cw.joblib: 230.2 MB

RF binary: 200 trees, depth range 37-63, mean 46.2


## 6. Re-compute SHAP for tree models under each perturbation

All 2000 perturbed samples, all 3 perturbation types. This is fast for trees.

In [10]:
TREE_MODELS = {
    'rf_binary_cw':     {'fn': 'rf',  'n_classes': 2, 'load': lambda: joblib.load(MODELS_DIR / 'rf_binary_cw.joblib')},
    'xgb_binary_cw':    {'fn': 'xgb', 'n_classes': 2, 'load': lambda: joblib.load(MODELS_DIR / 'xgb_binary_cw.joblib')},
    'rf_5class_smote':  {'fn': 'rf',  'n_classes': 5, 'load': lambda: joblib.load(MODELS_DIR / 'rf_5class_smote.joblib')},
    'xgb_5class_smote': {'fn': 'xgb', 'n_classes': 5, 'load': lambda: joblib.load(MODELS_DIR / 'xgb_5class_smote.joblib')},
}

# UNSW RF trees are extremely deep (mean depth 46, max 63), making TreeExplainer slow
# per-sample. We subsample with a per-class minimum to ensure all 5 classes are represented
# (the natural test partition has only ~1.3% U2R, which is too few even at n=200).
TREE_STAB_N_TARGET = 200
MIN_PER_CLASS = 15

np.random.seed(42)
selected = []
for cid in range(5):
    available = np.where(y_shap_5class == cid)[0]
    n_proportional = int(TREE_STAB_N_TARGET * len(available) / len(y_shap_5class))
    n_take = max(MIN_PER_CLASS, n_proportional)
    n_take = min(n_take, len(available))
    selected.extend(np.random.choice(available, n_take, replace=False).tolist())

tree_local = np.sort(np.array(selected))
np.save(STAB_DIR / 'tree_subsample_local_idx.npy', tree_local)

print(f'Tree stability subsample: {len(tree_local)} samples')
for cid in range(5):
    n = (y_shap_5class[tree_local] == cid).sum()
    print(f'  {FIVE_CLASS_NAMES[cid]:8s}: {n:>3d}')

def shap_rf_fast(model, X):
    explainer = shap.TreeExplainer(model)
    sv = explainer.shap_values(X, check_additivity=False)
    if isinstance(sv, list):
        sv = np.stack(sv, axis=-1)
    elif sv.ndim == 3 and sv.shape[0] != X.shape[0]:
        sv = np.transpose(sv, (1, 2, 0))
    elif sv.ndim == 2:
        sv = sv[..., np.newaxis]
    return sv

# Remove any stale shap files from previous attempts
import os
for name in TREE_MODELS:
    for pname in ['gaussian', 'fgsm', 'pgd']:
        f = STAB_DIR / f'{name}_{pname}_shap.npy'
        if f.exists():
            os.remove(f)
            print(f'Removed: {f.name}')

# Compute tree SHAP for all 12 (model, perturbation) combinations
for name, spec in TREE_MODELS.items():
    model = spec['load']()
    for pname, X_pert_full in PERTURBATIONS.items():
        out_path = STAB_DIR / f'{name}_{pname}_shap.npy'
        X_pert_sub = X_pert_full[tree_local]
        t0 = time.time()
        if spec['fn'] == 'xgb':
            sv = shap_xgb(model, X_pert_sub, spec['n_classes'])
        else:
            sv = shap_rf_fast(model, X_pert_sub)
        np.save(out_path, sv)
        print(f'  {name} / {pname}: shape={sv.shape}  time={time.time()-t0:.1f}s')

Tree stability subsample: 215 samples
  Normal  : 116
  DoS     :  15
  Probe   :  15
  R2L     :  56
  U2R     :  13
  rf_binary_cw / gaussian: shape=(215, 196, 2)  time=597.7s
  rf_binary_cw / fgsm: shape=(215, 196, 2)  time=585.8s
  rf_binary_cw / pgd: shape=(215, 196, 2)  time=589.8s
  xgb_binary_cw / gaussian: shape=(215, 196, 2)  time=0.2s
  xgb_binary_cw / fgsm: shape=(215, 196, 2)  time=0.1s
  xgb_binary_cw / pgd: shape=(215, 196, 2)  time=0.1s
  rf_5class_smote / gaussian: shape=(215, 196, 5)  time=1460.3s
  rf_5class_smote / fgsm: shape=(215, 196, 5)  time=1433.1s
  rf_5class_smote / pgd: shape=(215, 196, 5)  time=1456.1s
  xgb_5class_smote / gaussian: shape=(215, 196, 5)  time=0.5s
  xgb_5class_smote / fgsm: shape=(215, 196, 5)  time=0.3s
  xgb_5class_smote / pgd: shape=(215, 196, 5)  time=0.2s


In [13]:
import numpy as np

print(f'Length of y_shap_5class: {len(y_shap_5class)}')
print(f'U2R count in y_shap_5class: {(y_shap_5class == 4).sum()}')

# Check what np.where returns for U2R
available = np.where(y_shap_5class == 4)[0]
print(f'Available U2R indices: {len(available)}')

# Confirm the random seed wasn't overridden
print(f'\nSeed check — should give same first U2R index every time:')
np.random.seed(42)
test = np.random.choice(available, min(3, len(available)), replace=False)
print(f'  First 3 U2R picks at seed 42: {sorted(test)}')

# Check the cell that ran most recently — what did dnn_local actually pick?
print(f'\nCurrent dnn_local:')
print(f'  Total samples: {len(dnn_local)}')
for cid in range(5):
    n = (y_shap_5class[dnn_local] == cid).sum()
    print(f'  Class {cid}: {n}')

Length of y_shap_5class: 2000
U2R count in y_shap_5class: 13
Available U2R indices: 13

Seed check — should give same first U2R index every time:
  First 3 U2R picks at seed 42: [np.int64(22), np.int64(1224), np.int64(1425)]

Current dnn_local:
  Total samples: 500
  Class 0: 292
  Class 1: 32
  Class 2: 33
  Class 3: 140
  Class 4: 3


In [14]:
# Show the actual cell currently producing dnn_local
import inspect
# Check the current state - this just confirms what variables exist
print(f'dnn_local first 20 values: {sorted(dnn_local)[:20]}')
print(f'Their classes: {y_shap_5class[sorted(dnn_local)[:20]]}')

dnn_local first 20 values: [np.int64(3), np.int64(10), np.int64(19), np.int64(26), np.int64(31), np.int64(32), np.int64(33), np.int64(42), np.int64(45), np.int64(48), np.int64(49), np.int64(55), np.int64(56), np.int64(61), np.int64(65), np.int64(66), np.int64(67), np.int64(68), np.int64(70), np.int64(74)]
Their classes: [0 2 3 3 1 3 1 3 4 2 3 3 3 1 1 3 3 3 1 1]


## 7. Re-compute SHAP for DNNs under each perturbation

Subsample to 500 stratified samples per perturbation to keep runtime manageable. The 500 indices are consistent across perturbations so direct comparison is possible.

In [15]:
# Force-build the DNN subsample with proper min-per-class flooring
DNN_STAB_N_TARGET = 500
MIN_PER_CLASS_DNN = 15

np.random.seed(42)
selected = []
for cid in range(5):
    available = np.where(y_shap_5class == cid)[0]
    n_proportional = int(DNN_STAB_N_TARGET * len(available) / len(y_shap_5class))
    n_take = max(MIN_PER_CLASS_DNN, n_proportional)
    n_take = min(n_take, len(available))
    print(f'  Class {cid}: available={len(available)}, proportional={n_proportional}, taking={n_take}')
    selected.extend(np.random.choice(available, n_take, replace=False).tolist())

dnn_local = np.sort(np.array(selected))
np.save(STAB_DIR / 'dnn_subsample_local_idx.npy', dnn_local)

print(f'\nDNN stability subsample size: {len(dnn_local)}')
for cid in range(5):
    n = (y_shap_5class[dnn_local] == cid).sum()
    print(f'  {FIVE_CLASS_NAMES[cid]:8s}: {n:>3d}')

  Class 0: available=1166, proportional=291, taking=291
  Class 1: available=129, proportional=32, taking=32
  Class 2: available=132, proportional=33, taking=33
  Class 3: available=560, proportional=140, taking=140
  Class 4: available=13, proportional=3, taking=13

DNN stability subsample size: 509
  Normal  : 291
  DoS     :  32
  Probe   :  33
  R2L     : 140
  U2R     :  13


In [16]:
from sklearn.model_selection import train_test_split

DNN_STAB_N_TARGET = 500
MIN_PER_CLASS_DNN = 15

np.random.seed(42)
selected = []
for cid in range(5):
    available = np.where(y_shap_5class == cid)[0]
    n_proportional = int(DNN_STAB_N_TARGET * len(available) / len(y_shap_5class))
    n_take = max(MIN_PER_CLASS_DNN, n_proportional)
    n_take = min(n_take, len(available))
    selected.extend(np.random.choice(available, n_take, replace=False).tolist())

dnn_local = np.sort(np.array(selected))
np.save(STAB_DIR / 'dnn_subsample_local_idx.npy', dnn_local)

print(f'DNN stability subsample size: {len(dnn_local)}')
for cid in range(5):
    n = (y_shap_5class[dnn_local] == cid).sum()
    print(f'  {FIVE_CLASS_NAMES[cid]:8s}: {n:>3d}')

# Backgrounds — reuse from Notebook 04
bg_binary_idx = np.load(SHAP_DIR / 'bg_binary_idx.npy')
bg_5class_idx = np.load(SHAP_DIR / 'bg_5class_idx.npy')
X_bg_binary = X_train[bg_binary_idx]
X_bg_5class = X_train[bg_5class_idx]

DNN stability subsample size: 509
  Normal  : 291
  DoS     :  32
  Probe   :  33
  R2L     : 140
  U2R     :  13


In [17]:
DNN_SPECS = {
    'dnn_binary_cw':    {'n_classes': 2, 'bg': X_bg_binary, 'state': torch.load(MODELS_DIR / 'dnn_binary_cw.pt',    map_location=device)},
    'dnn_5class_smote': {'n_classes': 5, 'bg': X_bg_5class, 'state': torch.load(MODELS_DIR / 'dnn_5class_smote.pt', map_location=device)},
}

for name, spec in DNN_SPECS.items():
    for pname, X_pert_full in PERTURBATIONS.items():
        out_path = STAB_DIR / f'{name}_{pname}_shap.npy'
        if out_path.exists():
            print(f'  {name} / {pname}: already exists, skipping')
            continue
        X_pert_sub = X_pert_full[dnn_local]
        print(f'  {name} / {pname}... (5–8 min)')
        t0 = time.time()
        sv = shap_dnn(spec['state'], X_pert_sub, spec['bg'], N_FEATURES, spec['n_classes'])
        np.save(out_path, sv)
        print(f'    shape={sv.shape}  time={(time.time()-t0)/60:.1f} min')

  dnn_binary_cw / gaussian... (5–8 min)


    shape=(509, 196, 2)  time=5.0 min
  dnn_binary_cw / fgsm... (5–8 min)


    shape=(509, 196, 2)  time=5.0 min
  dnn_binary_cw / pgd... (5–8 min)


    shape=(509, 196, 2)  time=5.0 min
  dnn_5class_smote / gaussian... (5–8 min)


    shape=(509, 196, 5)  time=5.7 min
  dnn_5class_smote / fgsm... (5–8 min)


    shape=(509, 196, 5)  time=5.7 min
  dnn_5class_smote / pgd... (5–8 min)
    shape=(509, 196, 5)  time=5.7 min


## 8. Stability metrics — Jaccard top-10, Lipschitz median, F-Fidelity

In [19]:
def make_predictor(model_name):
    """Return a predict_proba(X) function for the given model name."""
    if model_name.startswith('rf_'):
        m = joblib.load(MODELS_DIR / f'{model_name}.joblib')
        return m.predict_proba
    elif model_name.startswith('xgb_'):
        m = joblib.load(MODELS_DIR / f'{model_name}.joblib')
        return m.predict_proba
    elif model_name.startswith('dnn_'):
        n_classes = DNN_SPECS[model_name]['n_classes']
        state = DNN_SPECS[model_name]['state']
        m = XIDSMLP(N_FEATURES, n_classes).to(device)
        m.load_state_dict(state)
        m.eval()
        def fn(arr):
            with torch.no_grad():
                t = torch.tensor(arr, dtype=torch.float32, device=device)
                return F.softmax(m(t), dim=1).cpu().numpy()
        return fn
    raise ValueError(model_name)

print('make_predictor defined.')

make_predictor defined.


In [22]:
from scipy.stats import spearmanr

TOP_K = 10

def jaccard_topk(sv_a, sv_b, k=TOP_K):
    """Per-sample Jaccard of top-k feature sets, averaged across samples."""
    imp_a = np.abs(sv_a).sum(axis=-1)
    imp_b = np.abs(sv_b).sum(axis=-1)
    n = imp_a.shape[0]
    j_sum = 0.0
    for i in range(n):
        top_a = set(np.argsort(imp_a[i])[::-1][:k])
        top_b = set(np.argsort(imp_b[i])[::-1][:k])
        union = top_a | top_b
        if not union:
            continue
        j_sum += len(top_a & top_b) / len(union)
    return j_sum / n

def lipschitz_median(sv_a, sv_b, x_a, x_b, eps=1e-8):
    """Median ||sv_b - sv_a|| / ||x_b - x_a|| across samples."""
    e_a = np.abs(sv_a).sum(axis=-1)
    e_b = np.abs(sv_b).sum(axis=-1)
    expl_norms  = np.linalg.norm(e_b - e_a, axis=1)
    input_norms = np.linalg.norm(x_b - x_a, axis=1)
    ratios = expl_norms / (input_norms + eps)
    return float(np.median(ratios))

def f_fidelity(model_predict_proba, x_a, sv_a, k=TOP_K, n_classes=None):
    """F-Fidelity: drop in predicted-class probability when top-k features are masked."""
    base_proba = model_predict_proba(x_a)
    base_pred  = base_proba.argmax(axis=1)
    base_conf  = base_proba[np.arange(len(x_a)), base_pred]

    imp = np.abs(sv_a).sum(axis=-1)

    x_masked = x_a.copy()
    for i in range(len(x_a)):
        top_idx = np.argsort(imp[i])[::-1][:k]
        x_masked[i, top_idx] = 0.0

    masked_proba = model_predict_proba(x_masked)
    masked_conf  = masked_proba[np.arange(len(x_a)), base_pred]
    return float((base_conf - masked_conf).mean())

def make_predictor(model_name):
    """Return a predict_proba(X) function for the given model name."""
    if model_name.startswith('rf_'):
        m = joblib.load(MODELS_DIR / f'{model_name}.joblib')
        return m.predict_proba
    elif model_name.startswith('xgb_'):
        m = joblib.load(MODELS_DIR / f'{model_name}.joblib')
        return m.predict_proba
    elif model_name.startswith('dnn_'):
        n_classes = DNN_SPECS[model_name]['n_classes']
        state = DNN_SPECS[model_name]['state']
        m = XIDSMLP(N_FEATURES, n_classes).to(device)
        m.load_state_dict(state)
        m.eval()
        def fn(arr):
            with torch.no_grad():
                t = torch.tensor(arr, dtype=torch.float32, device=device)
                return F.softmax(m(t), dim=1).cpu().numpy()
        return fn
    raise ValueError(model_name)

print('All helper functions defined.')

All helper functions defined.


In [23]:
ALL_MODELS = list(TREE_MODELS.keys()) + list(DNN_SPECS.keys())

stability_results = {}

for name in ALL_MODELS:
    print(f'\n=== {name} ===')
    stability_results[name] = {}

    # Load original SHAP (from Notebook 04)
    sv_orig_full = np.load(SHAP_DIR / f'{name}_shap.npy')

    # Trees use tree_local subsample (~215 samples); DNNs use dnn_local (~509 samples)
    if name.startswith('dnn_'):
        sv_orig = sv_orig_full[dnn_local]
        x_orig  = X_shap[dnn_local]
        idx_used = dnn_local
    elif name.startswith(('rf_', 'xgb_')):
        sv_orig = sv_orig_full[tree_local]
        x_orig  = X_shap[tree_local]
        idx_used = tree_local
    else:
        sv_orig = sv_orig_full
        x_orig  = X_shap
        idx_used = np.arange(len(X_shap))

    predict_fn = make_predictor(name)

    for pname in PERTURBATIONS:
        sv_pert = np.load(STAB_DIR / f'{name}_{pname}_shap.npy')
        x_pert  = PERTURBATIONS[pname][idx_used]

        j_topk = jaccard_topk(sv_orig, sv_pert, k=TOP_K)
        lip    = lipschitz_median(sv_orig, sv_pert, x_orig, x_pert)
        ff     = f_fidelity(predict_fn, x_orig, sv_orig, k=TOP_K)

        stability_results[name][pname] = {
            'jaccard_top10':    round(j_topk, 4),
            'lipschitz_median': round(lip, 4),
            'f_fidelity':       round(ff, 4),
            'n_samples':        int(sv_pert.shape[0]),
        }
        print(f'  {pname:8s}  Jaccard@10={j_topk:.4f}  Lipschitz={lip:.4f}  F-Fid={ff:.4f}')

with open(STAB_DIR / 'stability_metrics.json', 'w') as f:
    json.dump(stability_results, f, indent=2)
print('\nSaved stability_metrics.json')


=== rf_binary_cw ===
  gaussian  Jaccard@10=0.6143  Lipschitz=0.1324  F-Fid=0.3740
  fgsm      Jaccard@10=0.5963  Lipschitz=0.1388  F-Fid=0.3740
  pgd       Jaccard@10=0.6049  Lipschitz=0.1622  F-Fid=0.3740

=== xgb_binary_cw ===


/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [07:02:58] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


  gaussian  Jaccard@10=0.4139  Lipschitz=5.9271  F-Fid=0.4157
  fgsm      Jaccard@10=0.3881  Lipschitz=6.7790  F-Fid=0.4157
  pgd       Jaccard@10=0.3814  Lipschitz=7.8710  F-Fid=0.4157

=== rf_5class_smote ===
  gaussian  Jaccard@10=0.5484  Lipschitz=0.2420  F-Fid=0.4531
  fgsm      Jaccard@10=0.5147  Lipschitz=0.2666  F-Fid=0.4531
  pgd       Jaccard@10=0.5271  Lipschitz=0.3002  F-Fid=0.4531

=== xgb_5class_smote ===
  gaussian  Jaccard@10=0.4956  Lipschitz=7.0213  F-Fid=0.5810
  fgsm      Jaccard@10=0.4494  Lipschitz=8.1520  F-Fid=0.5810
  pgd       Jaccard@10=0.4708  Lipschitz=9.7949  F-Fid=0.5810

=== dnn_binary_cw ===
  gaussian  Jaccard@10=0.6668  Lipschitz=0.1335  F-Fid=0.6002
  fgsm      Jaccard@10=0.6650  Lipschitz=0.1385  F-Fid=0.6002
  pgd       Jaccard@10=0.6674  Lipschitz=0.1524  F-Fid=0.6002

=== dnn_5class_smote ===
  gaussian  Jaccard@10=0.7052  Lipschitz=0.1492  F-Fid=0.5345
  fgsm      Jaccard@10=0.6350  Lipschitz=0.2389  F-Fid=0.5345
  pgd       Jaccard@10=0.6281  L

## 9. Predict-proba helpers per model (for F-Fidelity)

In [24]:
def make_predictor(model_name):
    """Return a predict_proba(X) function for the given model name."""
    if model_name.startswith('rf_'):
        m = joblib.load(MODELS_DIR / f'{model_name}.joblib')
        return m.predict_proba
    elif model_name.startswith('xgb_'):
        m = joblib.load(MODELS_DIR / f'{model_name}.joblib')
        return m.predict_proba
    elif model_name.startswith('dnn_'):
        n_classes = DNN_SPECS[model_name]['n_classes']
        state = DNN_SPECS[model_name]['state']
        m = XIDSMLP(N_FEATURES, n_classes).to(device)
        m.load_state_dict(state)
        m.eval()
        def fn(arr):
            with torch.no_grad():
                t = torch.tensor(arr, dtype=torch.float32, device=device)
                return F.softmax(m(t), dim=1).cpu().numpy()
        return fn
    raise ValueError(model_name)

print('Predictor factory ready.')

Predictor factory ready.


## 10. Compute stability metrics for every (model, perturbation) pair

In [25]:
ALL_MODELS = list(TREE_MODELS.keys()) + list(DNN_SPECS.keys())

stability_results = {}

for name in ALL_MODELS:
    print(f'\n=== {name} ===')
    stability_results[name] = {}

    # Load original SHAP (from Notebook 04)
    sv_orig_full = np.load(SHAP_DIR / f'{name}_shap.npy')

    # Trees use 500-sample subsample (tree_local); DNNs use 500-sample subsample (dnn_local);
    # both are stratified from the same 2000-sample SHAP subsample so they're directly comparable
    if name.startswith('dnn_'):
        sv_orig = sv_orig_full[dnn_local]
        x_orig  = X_shap[dnn_local]
        idx_used = dnn_local
    elif name.startswith(('rf_', 'xgb_')):
        sv_orig = sv_orig_full[tree_local]
        x_orig  = X_shap[tree_local]
        idx_used = tree_local
    else:
        sv_orig = sv_orig_full
        x_orig  = X_shap
        idx_used = np.arange(len(X_shap))

    predict_fn = make_predictor(name)

    for pname in PERTURBATIONS:
        sv_pert = np.load(STAB_DIR / f'{name}_{pname}_shap.npy')
        x_pert  = PERTURBATIONS[pname][idx_used]

        j_topk = jaccard_topk(sv_orig, sv_pert, k=TOP_K)
        lip    = lipschitz_median(sv_orig, sv_pert, x_orig, x_pert)
        ff     = f_fidelity(predict_fn, x_orig, sv_orig, k=TOP_K)

        stability_results[name][pname] = {
            'jaccard_top10':    round(j_topk, 4),
            'lipschitz_median': round(lip, 4),
            'f_fidelity':       round(ff, 4),
            'n_samples':        int(sv_pert.shape[0]),
        }
        print(f'  {pname:8s}  Jaccard@10={j_topk:.4f}  Lipschitz={lip:.4f}  F-Fid={ff:.4f}')

with open(STAB_DIR / 'stability_metrics.json', 'w') as f:
    json.dump(stability_results, f, indent=2)
print('\nSaved stability_metrics.json')


=== rf_binary_cw ===
  gaussian  Jaccard@10=0.6143  Lipschitz=0.1324  F-Fid=0.3740
  fgsm      Jaccard@10=0.5963  Lipschitz=0.1388  F-Fid=0.3740
  pgd       Jaccard@10=0.6049  Lipschitz=0.1622  F-Fid=0.3740

=== xgb_binary_cw ===
  gaussian  Jaccard@10=0.4139  Lipschitz=5.9271  F-Fid=0.4157
  fgsm      Jaccard@10=0.3881  Lipschitz=6.7790  F-Fid=0.4157
  pgd       Jaccard@10=0.3814  Lipschitz=7.8710  F-Fid=0.4157

=== rf_5class_smote ===
  gaussian  Jaccard@10=0.5484  Lipschitz=0.2420  F-Fid=0.4531
  fgsm      Jaccard@10=0.5147  Lipschitz=0.2666  F-Fid=0.4531
  pgd       Jaccard@10=0.5271  Lipschitz=0.3002  F-Fid=0.4531

=== xgb_5class_smote ===
  gaussian  Jaccard@10=0.4956  Lipschitz=7.0213  F-Fid=0.5810
  fgsm      Jaccard@10=0.4494  Lipschitz=8.1520  F-Fid=0.5810
  pgd       Jaccard@10=0.4708  Lipschitz=9.7949  F-Fid=0.5810

=== dnn_binary_cw ===
  gaussian  Jaccard@10=0.6668  Lipschitz=0.1335  F-Fid=0.6002
  fgsm      Jaccard@10=0.6650  Lipschitz=0.1385  F-Fid=0.6002
  pgd       J

## 11. Summary table

In [ ]:
import pandas as pd

rows = []
for name in ALL_MODELS:
    for pname in PERTURBATIONS:
        r = stability_results[name][pname]
        rows.append({
            'Model':        name,
            'Perturbation': pname,
            'Jaccard@10':   r['jaccard_top10'],
            'Lipschitz':    r['lipschitz_median'],
            'F-Fidelity':   r['f_fidelity'],
            'n_samples':    r['n_samples'],
        })

stab_df = pd.DataFrame(rows)
stab_path = REPO / 'results/tables/unsw_stability_metrics.csv'
stab_df.to_csv(stab_path, index=False)

print(stab_df.to_string(index=False))
print(f'\nSaved: {stab_path}')

## 12. Stability heatmaps

In [ ]:
import matplotlib.pyplot as plt

metrics_to_plot = ['jaccard_top10', 'lipschitz_median', 'f_fidelity']
metric_labels   = ['Jaccard@10 (higher = better)', 'Lipschitz median (lower = better)', 'F-Fidelity (higher = better)']
metric_cmaps    = ['Greens', 'Reds_r', 'Greens']

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for ax, metric, label, cmap in zip(axes, metrics_to_plot, metric_labels, metric_cmaps):
    matrix = np.array([
        [stability_results[m][p][metric] for p in PERTURBATIONS]
        for m in ALL_MODELS
    ])
    im = ax.imshow(matrix, cmap=cmap, aspect='auto')
    ax.set_xticks(range(len(PERTURBATIONS)))
    ax.set_xticklabels(PERTURBATIONS.keys())
    ax.set_yticks(range(len(ALL_MODELS)))
    ax.set_yticklabels(ALL_MODELS, fontsize=9)
    ax.set_title(label, fontsize=11)
    for i in range(len(ALL_MODELS)):
        for j in range(len(PERTURBATIONS)):
            ax.text(j, i, f'{matrix[i, j]:.3f}', ha='center', va='center', fontsize=9, color='black')
    plt.colorbar(im, ax=ax, fraction=0.04)

plt.suptitle('UNSW-NB15 — Explanation stability under perturbation', fontsize=13, y=1.02)
plt.tight_layout()
fig_path = REPO / 'results/figures/unsw_stability_heatmaps.png'
plt.savefig(fig_path, dpi=180, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path}')

## 13. Commit and push

In [ ]:
os.chdir(PROJECT_DIR)
!git add notebooks/05_unsw_stability.ipynb \
         results/figures/unsw_stability_heatmaps.png \
         results/tables/unsw_stability_metrics.csv
!git commit -m "Notebook 05-UNSW: explanation stability under Gaussian/FGSM/PGD perturbation (C3)"
!git push origin main

---

## How to interpret the results

**Jaccard top-10** (higher = better, range 0–1)
- Close to 1.0 → top-10 features survive perturbation → robust explanation
- Close to 0.5 → about half the top features change → fragile
- Close to 0.0 → top-10 completely changes → explanation is meaningless under perturbation

**Lipschitz median** (lower = better, NOT comparable across architectures)
- XGBoost will show Lipschitz an order of magnitude larger than RF/DNN. This is NOT because XGBoost is fragile — it's because XGBoost's raw SHAP magnitudes are much larger. The Lipschitz ratio scales with explanation magnitude. Compare *within* an architecture across perturbations, not across architectures.

**F-Fidelity** (higher = better)
- High and stable across perturbations → SHAP correctly identifies prediction-driving features and this identification survives input perturbation
- Drops dramatically under FGSM/PGD → adversarial perturbation has shifted *which* features matter, undermining the explanation

## Expected patterns (from NSL-KDD and CIC-IDS2017)

- **DNN should be most stable** (Jaccard 0.7–0.9 across perturbations) — counterintuitive but consistent
- **XGBoost Lipschitz huge** — magnitude artifact, not fragility
- **Gaussian noise least disruptive**, **PGD most disruptive** — by design
- **Trees may show surprising stability under FGSM/PGD** — because the attacks were crafted against the DNN, trees see them as ~Gaussian-like noise

## Next step

**Notebook 06-UNSW** — Cross-model SHAP agreement using Krishna et al. 2022's six metrics. Uses the original (unperturbed) SHAP arrays from Notebook 04.